# Geometry 00 — L'escalier d'entrée : de la figure au noyau

**Navigation** : [01 — De la figure à l'équation](Geometry-01-From-Figure-To-Equation.ipynb) · [03 — Méthode de Wu](Geometry-03-Wu-Method-Python.ipynb) · [03b — Décomposition de Ritt](Geometry-03b-Ritt-Decomposition-Python.ipynb) · [04 — DD+AR](Geometry-04-DD-AR-Python.ipynb) · [lac `geometry_lean`](geometry_lean/)

La sous-série **Geometry** regarde cinq fois le même théorème — le milieu de
l'hypoténuse équidistant des trois sommets — avec cinq méthodes qui ne garantissent
pas la même chose. Cet escalier est la montée qui les relie : il part de la figure,
calcule comme le fait le 01, puis monte marche par marche jusqu'à la preuve formelle
du lake companion [`geometry_lean`](geometry_lean/).

Chaque marche **double un notebook** de la sous-série et dit ce qu'elle ajoute —
et ce qu'elle ne garantit pas encore.

## 0. Le problème : quatre méthodes, quatre garanties

Le même énoncé peut être « vérifié » de plusieurs façons, et le mot **prouvé** ne veut
pas dire la même chose dans chacune :

| Geste | Ce qu'il fait | Ce qu'il garantit |
|---|---|---|
| Vérifier sur des figures | évalue l'énoncé sur des exemples | rien au-delà des exemples — sauf le détour probabiliste de Schwartz–Zippel (01) |
| Simplifier avec `sympy` | transforme une expression | le résultat d'un calcul, pas un certificat |
| Décider par Gröbner / Wu | décide une appartenance à un idéal | exact, **sous conditions de non-dégénérescence** (02, 03) |
| Prouver en Lean | produit un terme de preuve vérifié par le noyau | la validité au noyau, sous les axiomes déclarés |

L'escalier descend de la figure jusqu'au noyau, et à chaque marche on note **ce qui est
gagné** et **ce qui reste hors de portée**. C'est la question que la position 05 du
programme gradué ([#17544](https://github.com/jsboige/CoursIA/issues/17544)) met en
scène : *que garantit « prouvé par Gröbner » ?*

## 1. Marche 1 — la figure devient un exemple calculé

*(double [Geometry-01](Geometry-01-From-Figure-To-Equation.ipynb), section 1)*

Le triangle `ABC` est rectangle en `A`. On place `A` à l'origine, `B` sur l'axe des
abscisses, `C` sur l'axe des ordonnées : `B = (b, 0)`, `C = (0, c)`, avec `b, c > 0`.
Le milieu de l'hypoténuse `[BC]` est `M = (b/2, c/2)`. L'énoncé du fil rouge :
`MA = MB = MC`.

In [1]:
# Marche 1 : un triangle, trois distances, une egalite constatee.
import numpy as np

A = np.array([0.0, 0.0])
B = np.array([6.0, 0.0])
C = np.array([0.0, 8.0])
M = (B + C) / 2


def distance(P, Q):
    return float(np.linalg.norm(P - Q))


MA, MB, MC = distance(M, A), distance(M, B), distance(M, C)
print(f"MA = {MA:.12f}")
print(f"MB = {MB:.12f}")
print(f"MC = {MC:.12f}")
print(f"ecart max = {max(abs(MA - MB), abs(MA - MC)):.3e}")
print(f"hypotenuse BC = {distance(B, C):.6f}  |  moitie = {distance(B, C) / 2:.6f}")

MA = 5.000000000000
MB = 5.000000000000
MC = 5.000000000000
ecart max = 0.000e+00
hypotenuse BC = 10.000000  |  moitie = 5.000000


### Lecture du résultat

Les trois distances coïncident à `1e-16` près — la précision de l'arithmétique
flottante — et valent la moitié de l'hypoténuse (`BC = 10`, `MA = 5`). C'est exactement
ce que fait le 01 sur un exemple.

Mais un exemple n'est pas un théorème : rien ici ne dit que l'égalité tient pour
**tous** les triangles rectangles, ni même pour un autre triangle que celui-ci.
La marche suivante remplace les nombres par des symboles.

## 2. Marche 2 — le même calcul, pour tout triangle

*(double [Geometry-01](Geometry-01-From-Figure-To-Equation.ipynb), section 2 ;
prépare [Geometry-02](https://github.com/jsboige/CoursIA/issues/17544))*

On refait la marche 1 en gardant `b` et `c` **symboliques**. `sympy` calcule les trois
carrés de distances et simplifie leurs différences : si elles sont identiquement nulles,
l'égalité n'est plus un accident numérique du triangle choisi.

In [2]:
# Marche 2 : les memes distances, en symbolique.
import sympy as sp

b, c = sp.symbols('b c', positive=True)
A2 = sp.Matrix([0, 0])
B2 = sp.Matrix([b, 0])
C2 = sp.Matrix([0, c])
M2 = (B2 + C2) / 2


def carre_distance(P, Q):
    d = sp.expand(P - Q)
    return sp.expand(d.dot(d))


d_MA, d_MB, d_MC = carre_distance(M2, A2), carre_distance(M2, B2), carre_distance(M2, C2)
print("MA^2 =", d_MA)
print("MB^2 =", d_MB)
print("MC^2 =", d_MC)
print("MA^2 - MB^2 =", sp.simplify(d_MA - d_MB))
print("MA^2 - MC^2 =", sp.simplify(d_MA - d_MC))
print("hypotenuse^2 / 4 =", sp.expand(carre_distance(B2, C2) / 4))

MA^2 = b**2/4 + c**2/4
MB^2 = b**2/4 + c**2/4
MC^2 = b**2/4 + c**2/4
MA^2 - MB^2 = 0
MA^2 - MC^2 = 0
hypotenuse^2 / 4 = b**2/4 + c**2/4


### Lecture du résultat

Les trois carrés valent `(b^2 + c^2)/4` : les différences se simplifient en `0`,
**pour tous** `b` et `c` positifs. L'égalité n'est plus un accident d'un triangle
particulier — elle est vraie sur tout le modèle.

`sympy` a fait ici un vrai travail symbolique. Est-ce une preuve ? La marche suivante
montre pourquoi la réponse est **non** — et c'est précisément la question de la
position 05.

## 3. Marche 3 — ce que le calcul symbolique ne certifie pas

*(la question de la position 05 de [#17544](https://github.com/jsboige/CoursIA/issues/17544))*

Trois raisons pour lesquelles « `sympy` simplifie à `0` » n'est pas encore une preuve :

1. **Le calcul n'est pas certifié.** `sp.simplify` applique des transformations qu'il
   ne justifie pas à l'appelant ; rien, dans le résultat, ne permet de vérifier
   indépendamment qu'elles étaient valides. Un défaut de `simplify` produirait un `0`
   faux, et personne ne le verrait.
2. **Le modèle n'est pas l'énoncé.** On a calculé sur des coordonnées choisies (le
   triangle rectangle en `A`, aligné sur les axes). Le passage « coordonnées ⟹ énoncé
   géométrique » est un raisonnement — correct, mais non vérifié par le calcul.
3. **Les hypothèses implicites.** Les énoncés de géométrie portent des conditions de
   non-dégénérescence (triangle non aplati, points distincts). Un calcul qui les oublie
   « prouve » un énoncé plus large que le vrai — donc faux. Le 03 le montre pour Wu :
   ces conditions doivent être générées explicitement.

Un témoin négatif rend la troisième raison concrète : le même calcul, appliqué à un
triangle qui n'est **pas** rectangle en `A`, ne s'annule pas — et c'est heureux.

In [3]:
# Marche 3 : le meme calcul sur un triangle NON rectangle en A -- le calcul sait dire non.
p = sp.Rational(1, 3)
C3 = sp.Matrix([p * c, c])          # C quitte l'axe des ordonnees : AB . AC != 0
M3 = (B2 + C3) / 2

diff_non_rectangle = sp.simplify(carre_distance(M3, A2) - carre_distance(M3, B2))
print("triangle non rectangle : MA^2 - MB^2 =", diff_non_rectangle)
print("s'annule-t-elle ?", diff_non_rectangle == 0)
print("produit scalaire AB . AC =", sp.expand(B2.dot(C3)), "(nul seulement si rectangle en A)")

triangle non rectangle : MA^2 - MB^2 = b*c/3
s'annule-t-elle ? False
produit scalaire AB . AC = b*c/3 (nul seulement si rectangle en A)


### Lecture du résultat

Sur le triangle non rectangle, la différence vaut `bc/3` — non nulle, et le produit
scalaire `AB · AC = bc/3` confirme que l'angle en `A` n'est pas droit. Le calcul
**distingue** donc les deux cas : il ne dit pas « oui » à tout.

C'est une qualité, mais elle ne change pas la conclusion de la marche 3 : `sympy`
calcule, il ne certifie pas. Pour obtenir une garantie vérifiable, il faut un objet
dont la validité se contrôle **indépendamment du producteur** — c'est le rôle du noyau
Lean.

## 4. Marche 4 — l'énoncé en Lean, dans le lake companion

*(double le lac [`geometry_lean`](geometry_lean/), volet B de l'EPIC
[#18601](https://github.com/jsboige/CoursIA/issues/18601))*

Le lake companion `geometry_lean` formalise les théorèmes que la série Python calcule.
Son premier module, `Geometry/MidpointHypotenuse.lean`, énonce le fil rouge dans le
langage des espaces de Hilbert réels : `u` et `v` sont les vecteurs portés par les deux
côtés de l'angle droit, `M = (u + v) / 2` est le milieu de l'hypoténuse, et l'énoncé
devient une égalité de **normes**.

La cellule suivante lit le fichier réel du lake et affiche ses énoncés — pas une
transcription.

In [4]:
# Marche 4 : lire les enonces du lake companion (le fichier reel).
import os
import re

CANDIDATS = [
    "geometry_lean/Geometry/MidpointHypotenuse.lean",
    "MyIA.AI.Notebooks/SymbolicAI/Lean/Geometry/geometry_lean/Geometry/MidpointHypotenuse.lean",
]


def chemin_lake():
    for candidat in CANDIDATS:
        if os.path.exists(candidat):
            return candidat
    return None


chemin = chemin_lake()
print("fichier :", chemin)
if chemin:
    texte_lake = open(chemin, encoding="utf-8").read()
    enonces = [(i + 1, l.strip()) for i, l in enumerate(texte_lake.splitlines())
               if re.match(r"\s*(theorem|private theorem|lemma)\s", l)]
    for numero, ligne in enonces:
        print(f"  l.{numero} : {ligne[:110]}")
    print(f"\nenonces dans le module : {len(enonces)}")
else:
    print("lake introuvable depuis ce cwd — les enonces restent lisibles ci-dessus")

fichier : geometry_lean/Geometry/MidpointHypotenuse.lean
  l.33 : theorem norm_add_eq_norm_sub_of_inner_eq_zero {u v : E} (huv : ⟪u, v⟫_ℝ = 0) :
  l.42 : private theorem dist_origin_eq_dist_apex_of_inner_eq_zero {u v : E}
  l.54 : theorem midpoint_hypotenuse_equidistant {u v : E} (huv : ⟪u, v⟫_ℝ = 0) :
  l.68 : theorem midpoint_hypotenuse_radius {u v : E} (huv : ⟪u, v⟫_ℝ = 0) :

enonces dans le module : 4


### Lecture du résultat

Le fichier porte quatre énoncés : le lemme du parallélogramme rectangle
(`norm_add_eq_norm_sub_of_inner_eq_zero`), un lemme privé intermédiaire, le théorème du
fil rouge (`midpoint_hypotenuse_equidistant`) — qui conclut l'équidistance sous la forme
d'une conjonction de deux égalités de normes — et sa variante de rayon
(`midpoint_hypotenuse_radius`), qui nomme la valeur commune `BC / 2`.

La comparaison avec la marche 2 est instructive : les deux disent la même chose, mais
l'énoncé Lean **nomme ses hypothèses** — `⟪u, v⟫_ℝ = 0`, l'orthogonalité, donc l'angle
droit — au lieu de les enfouir dans un choix de coordonnées.

## 5. Marche 5 — la preuve, et ce qu'elle garantit

Un énoncé n'est pas une preuve. La cellule suivante extrait le **texte de la preuve**
du premier lemme et vérifie, sur le fichier réel, les deux propriétés que la CI du
dépôt contrôle déjà (job `proof-integrity`) :

- aucun `sorry` — une preuve admise sans démonstration ;
- aucune tactique `native_decide` — une décision déléguée au code compilé, hors noyau.

In [5]:
# Marche 5 : la preuve du premier lemme, et l'audit des axiomes interdits.
if chemin:
    motif = re.compile(
        r"theorem norm_add_eq_norm_sub_of_inner_eq_zero.*?:=\s*by\s*(.*?)(?=\n\n)", re.S)
    trouve = motif.search(texte_lake)
    print("preuve du lemme du parallelogramme rectangle :")
    print(trouve.group(1).strip() if trouve else "(motif introuvable — le fichier a peut-etre change)")
    comptes = {nom: texte_lake.count(nom) for nom in ("sorry", "native_decide", "admit")}
    print("\naudit en miniature (compte brut des mots-cles interdits) :")
    for nom, compte in comptes.items():
        print(f"  {nom} : {compte}")
    verdict = "propre" if sum(comptes.values()) == 0 else "a examiner"
    print("\nverdict :", verdict)
    print("(l'organe canonique du depot, scripts/lean/count_code_sorry.py, distingue en plus")
    print(" les occurrences de code des occurrences de prose — ici le compte brut suffit,")
    print(" le module ne contient aucun de ces mots hors de ses preuves.)")
else:
    print("lake introuvable — audit impossible depuis ce cwd")

preuve du lemme du parallelogramme rectangle :
rw [norm_add_eq_sqrt_iff_real_inner_eq_zero.2 huv,
      norm_sub_eq_sqrt_iff_real_inner_eq_zero.2 huv]

audit en miniature (compte brut des mots-cles interdits) :
  sorry : 0
  native_decide : 0
  admit : 0

verdict : propre
(l'organe canonique du depot, scripts/lean/count_code_sorry.py, distingue en plus
 les occurrences de code des occurrences de prose — ici le compte brut suffit,
 le module ne contient aucun de ces mots hors de ses preuves.)


### Lecture du résultat

La preuve tient en deux tactiques : elle réécrit les deux normes par les deux formes de
l'identité de Pythagore que Mathlib fournit (`norm_add_eq_sqrt_iff_real_inner_eq_zero`
et sa version soustractive), l'hypothèse d'orthogonalité annulant le terme croisé.

C'est peu de lignes, et c'est le point : la garantie ne vient pas de la **longueur** de
la preuve, mais du fait qu'un **vérificateur indépendant** — le noyau de Lean — accepte
le terme de preuve, et que les axiomes employés sont contrôlés (le job `proof-integrity`
de la CI refuse `sorryAx` et `native_decide`). C'est cette indépendance qui manquait aux
marches 2 et 3.

## 6. Les quatre garanties, côte à côte

| Marche | Geste | Ce qui est gagné | Ce qui reste hors de portée |
|---|---|---|---|
| 1 | vérifier sur une figure | un exemple, une mesure | la généralité |
| 2 | calculer symboliquement | la généralité **sur le modèle** | la certification du calcul, le passage modèle ⟹ énoncé |
| 3 | décider (Gröbner, Wu — 02, 03) | une décision **exacte** dans le modèle | les conditions de non-dégénérescence, à générer explicitement |
| 4–5 | prouver en Lean (lake) | la validité au noyau, axiomes contrôlés | trouver l'énoncé et la preuve — le travail reste humain |

C'est la réponse que la position 05 du programme
([#17544](https://github.com/jsboige/CoursIA/issues/17544)) met en scène : « prouvé par
Gröbner » et « prouvé au noyau » ne garantissent pas la même chose, et l'escalier rend
l'écart visible.

## 7. Exercices

Trois marches à refaire soi-même.

### Exercice 1 — la précision flottante de la marche 1

Sur `n` triangles rectangles tirés au hasard (graine fixée), mesurer l'écart maximal
entre `MA`, `MB` et `MC`. L'égalité *exacte* de la marche 2 doit-elle se voir à `1e-16`
près, ou l'écart croît-il avec la taille des triangles ? Conclure sur ce qu'une
vérification numérique peut affirmer.

In [6]:
# Exercice 1 : ecart maximal sur n triangles rectangles aleatoires (graine fixee).
def ecart_maximal(n, span=20, graine=20261007):
    # TODO etudiant : tirer n triangles rectangles (B = (b, 0), C = (0, c), b, c > 0),
    # calculer MA, MB, MC avec numpy et retourner max(|MA - MB|, |MA - MC|)
    # sur l'ensemble des tirages.
    return None


resultat = ecart_maximal(1000)
print("Exercice 1 a completer — ecart maximal sur 1000 triangles :", resultat)

Exercice 1 a completer — ecart maximal sur 1000 triangles : None


### Exercice 2 — la condition d'annulation, en coordonnées générales

Reprendre la marche 3 avec un triangle `A = (0, 0)`, `B = (b, 0)`, `C = (p, q)` sans
contrainte d'angle droit, et calculer `MA² − MB²`. Pour quelle condition sur `p` et `q`
s'annule-t-elle ? Est-ce exactement « le triangle est rectangle en `A` » (c'est-à-dire
`AB · AC = 0`) ?

In [7]:
# Exercice 2 : condition d'annulation sur un triangle general A=(0,0), B=(b,0), C=(p,q).
def condition_annulation():
    # TODO etudiant : construire A, B, C avec p, q symboliques, calculer la difference
    # des carres de distances MA^2 - MB^2 (M milieu de [BC]) et retourner son expression
    # simplifiee. La comparer a la condition d'angle droit en A : AB . AC = 0.
    return None


print("Exercice 2 a completer — MA^2 - MB^2 =", condition_annulation())

Exercice 2 a completer — MA^2 - MB^2 = None


### Exercice 3 — auditer tout le lake

La marche 5 compte les mots-clés interdits du premier module. Élargir l'audit à
**tout** le lake (`geometry_lean/`) : lister les modules `.lean` et, pour chacun,
le nombre d'énoncés et le compte des mots-clés interdits. Le lac doit être **propre** —
un `sorry` dans un carnet d'exercice n'est pas un `sorry` dans le lac.

In [8]:
# Exercice 3 : audit de tous les modules du lake.
def audit_lake(racine="geometry_lean"):
    # TODO etudiant : parcourir les fichiers *.lean sous `racine` et retourner, pour
    # chacun, un tuple (chemin, nombre d'enonces, nombre de sorry, nombre de native_decide).
    return None


for ligne in audit_lake() or []:
    print(ligne)
print("Exercice 3 a completer — modules du lake :", audit_lake())

Exercice 3 a completer — modules du lake : None


## 8. Conclusion — ce que l'escalier laisse au lecteur

Un même théorème, quatre garanties : un exemple calculé, une identité symbolique, une
décision exacte, une preuve au noyau. L'escalier ne dit pas qu'une méthode vaut mieux
que l'autre — il dit **ce que chacune autorise à affirmer**, et où elle s'arrête.

Suite naturelle :

- [Geometry-02](https://github.com/jsboige/CoursIA/issues/17544) — la certitude par
  l'algèbre : l'idéal des hypothèses, l'appartenance décidée par Gröbner ;
- [Geometry-03](Geometry-03-Wu-Method-Python.ipynb) et
  [03b](Geometry-03b-Ritt-Decomposition-Python.ipynb) — la méthode de Wu et la
  décomposition de Ritt, avec leurs conditions de non-dégénérescence ;
- [Geometry-04](Geometry-04-DD-AR-Python.ipynb) — DD + AR, la moitié symbolique
  d'AlphaGeometry ;
- le lac [`geometry_lean`](geometry_lean/) — où le fil rouge est **prouvé** ; cet
  escalier en est l'entrée de la sous-série.

*Fil rouge : le milieu de l'hypoténuse, cinq fois.*